In [1]:
from datasets import load_dataset

In [2]:
ragbench_finqa = load_dataset("rungalileo/ragbench", "finqa", split="test")

In [3]:
ragbench_finqa["id"]

Column(['finqa_6345', 'finqa_7055', 'finqa_6634', 'finqa_6587', 'finqa_7027', ...])

In [4]:
ragbench_finqa[0]

{'id': 'finqa_6345',
 'question': 'what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?',
 'documents': ['stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . the graph assumes that the value of the investment in our common stock on january 2 , 2010 and in each index on december 31 , 2009 ( including reinvestment of dividends ) was $ 100 and tracks it each year thereafter on the last day of cadence 2019s fiscal year through january 3 , 2015 and , for each index , on the last day of the calendar comparison of 5 year cumulative total return* among cadence design systems , inc. , the nasdaq composite index , and s&p 400 information technology cadence design systems , inc . nasdaq composite s&p 400 information technology 12/28/13 1/3/151/1/11 12/31/11 12/

In [5]:
from langchain_core.documents import Document

In [6]:
def deduplicate_data(data):
    data_dict = {}
    for d in data:
        # print(d)
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            # print(d)
            # break
            data_dict[document] = {"docid":[d["id"]]}
    return data_dict

In [7]:
dedup = deduplicate_data(ragbench_finqa)

In [8]:
docs = [
    Document(
        
            metadata=v, 
            page_content=k
        
        
    )
    for k,v in dedup.items()
]

In [9]:
docs[:3]

[Document(metadata={'docid': ['finqa_6345', 'finqa_6941', 'finqa_6626', 'finqa_6720', 'finqa_6345', 'finqa_6941', 'finqa_6626', 'finqa_6720']}, page_content='stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . the graph assumes that the value of the investment in our common stock on january 2 , 2010 and in each index on december 31 , 2009 ( including reinvestment of dividends ) was $ 100 and tracks it each year thereafter on the last day of cadence 2019s fiscal year through january 3 , 2015 and , for each index , on the last day of the calendar comparison of 5 year cumulative total return* among cadence design systems , inc. , the nasdaq composite index , and s&p 400 information technology cadence design systems , inc . nasdaq composite s&p 400 information technology 12/28/13 1/3/151/1/11 12

In [10]:
len(docs)

380

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=200, separators=["\n\n", "\n", " ", ".", ","])
docs_chunks = text_splitter.split_documents(docs)

In [13]:
len(docs_chunks)

1909

In [14]:
docs_chunks[:2]

[Document(metadata={'docid': ['finqa_6345', 'finqa_6941', 'finqa_6626', 'finqa_6720', 'finqa_6345', 'finqa_6941', 'finqa_6626', 'finqa_6720']}, page_content='stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . the graph assumes that the value of the investment in our common stock on january 2 , 2010 and in each index on december 31 , 2009 ( including reinvestment of dividends ) was $ 100 and tracks it each year thereafter on the last day of cadence 2019s fiscal year through january 3 , 2015 and , for each index , on the last day of the calendar comparison of 5 year cumulative total return* among cadence design systems , inc. , the nasdaq composite index , and s&p 400 information technology cadence design systems , inc . nasdaq composite s&p 400 information technology 12/28/13 1/3/151/1/11 12

In [15]:
from collections import Counter


    

In [16]:
docs_count = Counter(chunk.page_content for chunk in docs_chunks)



In [17]:
import numpy as np
counts = list(docs_count.values())
counts

print(f"max_chunk_by_id : {max(counts)}")
print(f"min_chunk_by_id : {min(counts)}")
print(f"avg_chunk_by_id : {np.mean(counts):.2f}")



max_chunk_by_id : 2
min_chunk_by_id : 1
avg_chunk_by_id : 1.00


In [18]:
# embedding
chromadb_folder = "database"
db_name = "finance"
persist_directory = f"{chromadb_folder}/{db_name}"


In [19]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings



/var/folders/3f/z4mxt4h16g95cy326q6m_cqm0000gn/T/ipykernel_49984/1179322155.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [20]:
vector_db = Chroma.from_documents(documents=docs_chunks, 
                                  embedding=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5"),
                                  persist_directory=persist_directory)

/var/folders/3f/z4mxt4h16g95cy326q6m_cqm0000gn/T/ipykernel_49984/569640239.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5"),


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [52]:
print(f"Collection count: {vector_db._collection.count()}")

Collection count: 1909


In [40]:
# db_path = "database/finance"
# vector_db = Chroma(persist_directory=db_path, embedding_function=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [49]:
# retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [42]:
# from langchain_groq import ChatGroq
# from langchain_core.prompts import ChatPromptTemplate
# from dotenv import load_dotenv
# from langchain_core.output_parsers import StrOutputParser

In [43]:
# load_dotenv()

True

In [44]:


# eval_message = ["""I asked someone to answer a question based on one or more documents.
# Your task is to review their response and assess whether or not each sentence
# in that response is supported by text in the documents. And if so, which
# sentences in the documents provide that support. You will also tell me which
# of the documents contain useful information for answering the question, and
# which of the documents the answer was sourced from.
# Here are the documents, each of which is split into sentences. Alongside each
# sentence is associated key, such as ’a0.’ or ’b0.’ that you can use to refer
# to it:
# ‘‘‘
# {documents}
# ‘‘‘
# The question was:
# ‘‘‘
# {question}
# ‘‘‘
# Here is their response:
# ‘‘‘
# {answer}
# ‘‘‘
# You must respond with a JSON object matching this schema:
# ‘‘‘
# {{
# "relevance_explanation": string,
# "all_relevant_sentence_keys": [string],
# "overall_supported_explanation": string,
# "overall_supported": boolean,
# "sentence_support_information": [
# {{
# "response_sentence_key": string,
# "explanation": string,
#     "supporting_sentence_keys": [string],
# "fully_supported": boolean
# }}
# ],
# "all_utilized_sentence_keys": [string]
# }}
# ‘‘‘
# The relevance_explanation field is a string explaining which documents
# contain useful information for answering the question. Provide a step-by-step
# breakdown of information provided in the documents and how it is useful for
# answering the question.
# The all_relevant_sentence_keys field is a list of all document sentences keys
# (e.g. ’a0’) that are revant to the question. Include every sentence that is
# useful and relevant to the question, even if it was not used in the response,
# or if only parts of the sentence are useful. Ignore the provided response when
# making this judgement and base your judgement solely on the provided documents
# and question. Omit sentences that, if removed from the document, would not
# impact someone’s ability to answer the question.
# The overall_supported_explanation field is a string explaining why the response
# *as a whole* is or is not supported by the documents. In this field, provide a
# step-by-step breakdown of the claims made in the response and the support (or
# lack thereof) for those claims in the documents. Begin by assessing each claim
# separately, one by one; don’t make any remarks about the response as a whole
# until you have assessed all the claims in isolation.
# The overall_supported field is a boolean indicating whether the response as a
# whole is supported by the documents. This value should reflect the conclusion
# you drew at the end of your step-by-step breakdown in overall_supported_explanation.
# In the sentence_support_information field, provide information about the support
# *for each sentence* in the response.
# The sentence_support_information field is a list of objects, one for each sentence
# in the response. Each object MUST have the following fields:
# - response_sentence_key: a string identifying the sentence in the response.
# This key is the same as the one used in the response above.
# - explanation: a string explaining why the sentence is or is not supported by the
# documents.
# - supporting_sentence_keys: keys (e.g. ’a0’) of sentences from the documents that
# support the response sentence. If the sentence is not supported, this list MUST
# be empty. If the sentence is supported, this list MUST contain one or more keys.
# In special cases where the sentence is supported, but not by any specific sentence,
# you can use the string "supported_without_sentence" to indicate that the sentence
# is generally supported by the documents. Consider cases where the sentence is
# expressing inability to answer the question due to lack of relevant information in
# the provided contex as "supported_without_sentence". In cases where the sentence
# is making a general statement (e.g. outlining the steps to produce an answer, or
# summarizing previously stated sentences, or a transition sentence), use the
# string "general". In cases where the sentence is correctly stating a well-known fact,
# like a mathematical formula, use the string "well_known_fact". In cases where the
# sentence is performing numerical reasoning (e.g. addition, multiplication), use
# the string "numerical_reasoning".
# - fully_supported: a boolean indicating whether the sentence is fully supported by
# the documents.
# - This value should reflect the conclusion you drew at the end of your step-by-step
# breakdown in explanation.
# - If supporting_sentence_keys is an empty list, then fully_supported must be false.
# - Otherwise, use fully_supported to clarify whether everything in the response
# sentence is fully supported by the document text indicated in supporting_sentence_keys
# (fully_supported = true), or whether the sentence is only partially or incompletely
# supported by that document text (fully_supported = false).
# The all_utilized_sentence_keys field is a list of all sentences keys (e.g. ’a0’) that
# were used to construct the answer. Include every sentence that either directly supported
# the answer, or was implicitly used to construct the answer, even if it was not used
# in its entirety. Omit sentences that were not used, and could have been removed from
# the documents without affecting the answer.
# You must respond with a valid JSON string. Use escapes for quotes, e.g. ‘\\"‘, and
# newlines, e.g. ‘\\n‘. Do not write anything before or after the JSON string. Do not
# wrap the JSON string in backticks like ‘‘‘ or ‘‘‘json.
# As a reminder: your task is to review the response and assess which documents contain
# useful information pertaining to the question, and how each sentence in the response
# is supported by the text in the documents.
# """]

In [45]:
# def simple_rag_system(question: str, eval_message: str):
#     relevant_docs = retriever.invoke(question)
#     # print(relevant_docs)
#     sent_list = list()
#     for i, d in enumerate(relevant_docs):
#         index_char = chr(97+i)
#         for j, s in enumerate(d.page_content.split(".")):
#             sent_list.append([index_char+str(j), s])
#     # print(sent_list[:5])

    
            
        
#     context = "\n".join([d.page_content for d in relevant_docs])
#     # print(context)
    
#     messages = [
#     ("system", """You are a financial expert. You are given a question and a list of documents and need to
#     answer the question. Answer the question only based on these documents. These
#     documents can help you answer the question: {context}. If you are not sure about the
#     answer, you can say 'I don't know' or 'I don't know the answer to that question.'"""),
#     ("human", "{question}"),
#     ]
#     prompt = ChatPromptTemplate(messages=messages)
#     model = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
#     chain = prompt | model | StrOutputParser()
#     answer = chain.invoke({"question":question, "context":context})
#     prompt2 = ChatPromptTemplate(messages=eval_message)
#     model2 = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
#     chain2 = prompt2 | model2 | StrOutputParser()
#     answer2 = chain2.invoke({"documents":sent_list, "question":question, "answer":answer})
#     print(answer)
#     print(answer2)
#     # return answer

In [51]:
# query = "what is the maximum depreciation rate that can be used for furniture fixtures and equipment?"
# # what was the lowest effective tax rate in the three year period? 
# # response: The lowest effective tax rate in the three year period was 27% (27%) in 2005. 
# # Q: what is the maximum depreciation rate that can be used for furniture fixtures and equipment? 
# # response: The maximum depreciation rate that can be used for furniture fixtures and equipment is 10 years. 
# #  docid: finqa_6410
# # Q: what portion of the total 2015 restructuring programs is related to termination benefits? 
# # what portion of anios' purchasing price is related to goodwill? 
# # Is there a fee increase or consent requirement, etc. if one party’s use of the product/services exceeds certain threshold?


# print(simple_rag_system(query, eval_message))

According to the provided documents, the estimated useful lives for furniture, fixtures, and equipment are 3-10 years. 

To determine the maximum depreciation rate, we need to use the minimum useful life, which is 3 years. 

The maximum depreciation rate would be 100% / 3 years = 33.33% per year.
```
{
  "relevance_explanation": "The documents that contain useful information for answering the question are documents 'a' and 'b'. Document 'a' provides information about the estimated useful lives of furniture, fixtures, and equipment, which is relevant to determining the maximum depreciation rate. Document 'b' also provides information about the depreciation of property and equipment, but it does not specifically mention furniture, fixtures, and equipment. However, document 'a' is the most relevant to the question. The information provided in document 'a' is useful because it gives the estimated useful lives of furniture, fixtures, and equipment, which can be used to calculate the maximum

In [61]:
# for d in ragbench_finqa:
#     if d["question"] == "as of december 2007 what was the percent of the square footage in alpharetta georgia not yet leased ":
#         # print(f"Q: {d["question"]} \nresponse: {d["response"]} \n docid: {d["id"]}")
#         print(d)

TypeError: string indices must be integers, not 'str'

In [42]:
# print(vector_db._collection.count())

858


In [69]:
# for d in ragbench_finqa[0]:
#     print(d)

id
question
documents
response
generation_model_name
annotating_model_name
dataset_name
documents_sentences
response_sentences
sentence_support_information
unsupported_response_sentence_keys
adherence_score
overall_supported_explanation
relevance_explanation
all_relevant_sentence_keys
all_utilized_sentence_keys
trulens_groundedness
trulens_context_relevance
ragas_faithfulness
ragas_context_relevance
gpt3_adherence
gpt3_context_relevance
gpt35_utilization
relevance_score
utilization_score
completeness_score


In [74]:
# adherence_score = [int(value) for value in ragbench_finqa["adherence_score"]]
# relevance_score = [value for value in ragbench_finqa["relevance_score"]]
# utilization_score = [value for value in ragbench_finqa["utilization_score"]]
# completeness_score = [value for value in ragbench_finqa["completeness_score"]]

In [76]:
# import numpy as np

In [78]:
# adherence_score_mean = np.mean(adherence_score)
# relevance_score_mean = np.mean(relevance_score)
# utilization_score_mean = np.mean(utilization_score)
# completeness_score_mean = np.mean(completeness_score)

# print(adherence_score_mean)
# print(adherence_score_mean)
# print(adherence_score_mean)
# print(adherence_score_mean)


np.float64(0.9145597210113339)

In [79]:
# import requests
# import os

# api_key = os.environ.get("GROQ_API_KEY")
# url = "https://api.groq.com/openai/v1/models"

# headers = {
#     "Authorization": f"Bearer {api_key}",
#     "Content-Type": "application/json"
# }

# response = requests.get(url, headers=headers)

# print(response.json())

{'object': 'list', 'data': [{'id': 'openai/gpt-oss-safeguard-20b', 'object': 'model', 'created': 1761708789, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'whisper-large-v3', 'object': 'model', 'created': 1693721698, 'owned_by': 'OpenAI', 'active': True, 'context_window': 448, 'public_apps': None, 'max_completion_tokens': 448}, {'id': 'canopylabs/orpheus-arabic-saudi', 'object': 'model', 'created': 1765926439, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'meta-llama/llama-4-scout-17b-16e-instruct', 'object': 'model', 'created': 1743874824, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'allam-2-7b', 'object': 'model', 'created': 1737672203, 'owned_by': 'SDAIA', 'active': True, 'context_window': 4096, 'public_apps': None, 'max_completion_tokens': 4096}, 